# Anomaly detection with an autoencoder: finding damaged 5s



Companion notebook of the guidebook **"Anomaly detection with your autoencoder"**. It continues the previous lab: same data (the digit 5), same vanilla autoencoder.

**The idea in one sentence.** An autoencoder trained only on *normal* images learns to rebuild normal images. Show it a damaged image: it rebuilds what a normal 5 *should* look like, so the **difference between its output and its input** reveals the damage.

| Step | What happens |
|---|---|
| 1 | **given:** the data, ready to use: normal 5s to train, normal 5s to calibrate, and a test set where half of the 5s have a piece of stroke erased |
| 2 | **given:** train the autoencoder of the previous lab on normal 5s only |
| 3 | *you build:* the error maps: squared error and **signed difference** $\hat x - x$ |
| 4 | *you build:* a first anomaly score (mean error) and its AUC |
| 5 | *you build:* a threshold chosen on normal images, and the detection results |
| 6 | *you build:* a better score: the **missing ink** |
| 7 | *you build:* the picture of the anomalous region on each image |

After each step there is a **check cell**: run it, and only move on when it prints `OK`.

## Step 0. Setup (nothing to do)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
DIGIT = 5
LATENT_DIM = 32
BATCH_SIZE = 128
EPOCHS = 20
LEARNING_RATE = 1e-3
N_VAL = 500             # normal training images kept aside to choose the threshold
PATCH_SIZE = 7          # side of the square erased from the anomalous digits
ANOMALY_FRACTION = 0.5  # fraction of the test set that gets damaged
WINDOW = 5              # size of the local window used in Step 6

### Code from the previous lab (given)
`load_digit`, the `Autoencoder` and the training functions are exactly those of lab 05. The only change: `train` shows reconstructions only if `show=True`.

In [ ]:
def load_digit(digit=5, train=True):
    dataset = torchvision.datasets.MNIST(root="data", train=train, download=True)
    return (dataset.data[dataset.targets == digit].float() / 255.0).unsqueeze(1)


class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 784), nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train(); total = 0.0
    for (x,) in loader:
        x = x.to(device)
        loss = loss_fn(model(x), x)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval(); total = 0.0
    for (x,) in loader:
        x = x.to(device)
        total += loss_fn(model(x), x).item()
    return total / len(loader)


def show_row(images, n=8, title="", cmap="gray", vmin=0, vmax=1):
    fig, axes = plt.subplots(1, n, figsize=(1.5 * n, 1.8))
    for i in range(n):
        axes[i].imshow(images[i, 0].detach().cpu(), cmap=cmap, vmin=vmin, vmax=vmax)
        axes[i].axis("off")
    fig.suptitle(title)
    plt.show()


def train(model, train_loader, test_loader, epochs=10, lr=1e-3):
    history = {"train": [], "test": []}
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        history["train"].append(train_one_epoch(model, train_loader, optimizer, loss_fn))
        history["test"].append(evaluate(model, test_loader, loss_fn))
        print(f"epoch {epoch:2d}/{epochs}  train loss {history['train'][-1]:.4f}  val loss {history['test'][-1]:.4f}")
    return history

## Step 1. The data (given, nothing to write)

One function builds everything you need. Read it once: it is short, and knowing how the anomalies were made helps to interpret the results.

| tensor | shape | content | used for |
|---|---|---|---|
| `x_train` | (4921, 1, 28, 28) | normal 5s | training the autoencoder |
| `x_val` | (500, 1, 28, 28) | normal 5s, never trained on | choosing the thresholds (Steps 5 and 7) |
| `x_test` | (892, 1, 28, 28) | test 5s, **half of them damaged** | measuring how well we detect |
| `y_test` | (892,) | 1 = damaged (anomaly), 0 = normal | evaluation only |
| `m_test` | (892, 1, 28, 28) | 1 inside the erased square, 0 elsewhere | evaluation of the localisation only |

**How an anomaly is made.** A pixel of the stroke is chosen at random, and the `PATCH_SIZE x PATCH_SIZE` square centred on it is set to black. Centring on the stroke guarantees that some ink is really removed (a square on the black background would change nothing).

**Rule of the game.** `y_test` and `m_test` are the answers: the detector must never use them, only the evaluation does.

In [ ]:
def make_anomaly_data(digit=DIGIT, n_val=N_VAL, patch_size=PATCH_SIZE,
                      anomaly_fraction=ANOMALY_FRACTION, seed=0):
    """Build the data of the lab.

    Returns x_train, x_val, x_test, y_test, m_test:
      x_train (N_train, 1, 28, 28)  normal training images        -> train the autoencoder
      x_val   (n_val, 1, 28, 28)    normal images, never trained  -> choose the thresholds
      x_test  (N_test, 1, 28, 28)   test images, a fraction of them damaged
      y_test  (N_test,)             1 = damaged (anomaly), 0 = normal
      m_test  (N_test, 1, 28, 28)   1 inside the erased square, 0 elsewhere
    """
    g = torch.Generator().manual_seed(seed)

    # 1. normal images: split the training set into train / validation
    x_all = load_digit(digit, train=True)
    perm = torch.randperm(len(x_all), generator=g)
    x_val, x_train = x_all[perm[:n_val]], x_all[perm[n_val:]]

    # 2. test images: damage a random fraction of them
    x_test = load_digit(digit, train=False)
    n = len(x_test)
    y_test = torch.zeros(n, dtype=torch.long)
    m_test = torch.zeros_like(x_test)
    damaged = torch.randperm(n, generator=g)[: int(anomaly_fraction * n)]
    for i in damaged.tolist():
        ys, xs = torch.where(x_test[i, 0] > 0.5)                    # pixels of the stroke
        k = torch.randint(len(ys), (1,), generator=g).item()        # one of them = centre of the patch
        y0 = min(max(ys[k].item() - patch_size // 2, 0), 28 - patch_size)   # keep the square inside
        x0 = min(max(xs[k].item() - patch_size // 2, 0), 28 - patch_size)
        x_test[i, 0, y0:y0 + patch_size, x0:x0 + patch_size] = 0.0  # erase the ink
        m_test[i, 0, y0:y0 + patch_size, x0:x0 + patch_size] = 1.0  # remember where
        y_test[i] = 1
    return x_train, x_val, x_test, y_test, m_test


x_train, x_val, x_test, y_test, m_test = make_anomaly_data()

In [ ]:
# ---- look at the data ----
x_test_clean = load_digit(DIGIT, train=False)       # the same test images, before damage
print("x_train", tuple(x_train.shape), "| x_val", tuple(x_val.shape), "| x_test", tuple(x_test.shape))
print("test set:", int(y_test.sum()), "damaged and", int((y_test == 0).sum()), "normal images")
removed_ink = (x_test_clean * m_test).flatten(1).sum(1)[y_test == 1]
print(f"on average {removed_ink.mean():.1f} pixels of ink removed per damaged digit")
assert torch.equal(x_test[y_test == 0], x_test_clean[y_test == 0]), "normal test images are untouched"
ia = torch.where(y_test == 1)[0][:8]
show_row(x_test_clean[ia], title="before damage")
show_row(x_test[ia], title="damaged test images (anomalies)")
show_row(m_test[ia], title="ground-truth masks (m_test)")
show_row(x_test[torch.where(y_test == 0)[0][:8]], title="normal test images")
print("Step 1 OK")

## Step 2. Train on normal 5s only (code given)
Nothing to write: we reuse the training code of the previous lab. Note what is used for training (only `x_train`) and for validation (`x_val`, also normal).

In [ ]:
train_loader = DataLoader(TensorDataset(x_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val), batch_size=BATCH_SIZE)
torch.manual_seed(0)
model = Autoencoder(LATENT_DIM).to(device)
history = train(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE)
model.eval()

**Question.** Why must the training set contain **no** damaged digits? What would happen if half of the training 5s had holes?



## Step 3. Error maps: where does the reconstruction disagree with the input?

Write `error_maps(model, images)`. It returns three tensors of shape `(N, 1, 28, 28)`:
* `recon`: the reconstructions $\hat x$,
* `sq_error`: the squared error per pixel $(x - \hat x)^2$,
* `signed`: the **signed difference** $\hat x - x$. It is positive where the model expected ink that is **missing**, negative where there is ink the model did **not** expect.

In [ ]:
@torch.no_grad()
def error_maps(model, images):
    model.eval()
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# ---- check step 3 ----
recon, sq_err, signed = error_maps(model, x_test)
assert recon.shape == sq_err.shape == signed.shape == x_test.shape
assert torch.allclose(sq_err, signed ** 2, atol=1e-6), "squared error must equal the signed difference squared"
assert torch.allclose(recon - x_test, signed, atol=1e-6), "signed difference is recon - input"
ia = torch.where(y_test == 1)[0][:8]
show_row(x_test[ia], title="input (damaged)")
show_row(recon[ia], title="reconstruction")
show_row(sq_err[ia], title="squared error (x - x_hat)^2", cmap="magma", vmax=0.5)
show_row(signed[ia], title="signed difference x_hat - x   (red = missing ink, blue = unexpected ink)", cmap="bwr", vmin=-1, vmax=1)
show_row(m_test[ia], title="ground truth")
inn = torch.where(y_test == 0)[0][:8]
show_row(x_test[inn], title="NORMAL inputs")
show_row(sq_err[inn], title="their squared error", cmap="magma", vmax=0.5)
print("Step 3 OK")

**Question.** Look at the squared error of the **normal** images. Is it zero? Where is it largest?



## Step 4. A first anomaly score: the mean squared error

**5a.** Write `score_mean(sq_error)`: one number per image, the mean of its squared-error map. Return a 1-D tensor of length N.

**5b.** Compute `scores_v1` on the test set and its **ROC AUC** with `roc_auc_score(labels, scores)` from scikit-learn. The check cell draws the histograms of the scores of normal and anomalous images.

In [ ]:
def score_mean(sq_error):
    # YOUR CODE HERE
    raise NotImplementedError()

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# ---- check step 4 ----
def plot_score_histograms(scores, labels, title, threshold=None):
    plt.figure(figsize=(6, 3.5))
    bins = np.linspace(scores.min().item(), scores.max().item(), 40)
    plt.hist(scores[labels == 0].numpy(), bins=bins, alpha=0.6, label="normal")
    plt.hist(scores[labels == 1].numpy(), bins=bins, alpha=0.6, label="anomaly (patch erased)")
    if threshold is not None:
        plt.axvline(threshold, color="k", ls="--", label="threshold")
    plt.xlabel("anomaly score"); plt.ylabel("number of images"); plt.title(title); plt.legend(); plt.show()

assert scores_v1.shape == (len(x_test),)
print(f"AUC with the mean squared error: {auc_v1:.3f}")
plot_score_histograms(scores_v1, y_test, f"Score v1: mean squared error (AUC {auc_v1:.3f})")
print("Step 4 OK")

**Reading the AUC.** Pick one normal and one anomalous image at random: the AUC is the probability that the anomalous one gets the higher score. 0.5 is a coin toss, 1.0 is perfect.

## Step 5. Choosing a threshold without looking at anomalies

In real life we have no labelled anomalies to tune on. We choose the threshold using **normal images only**: the validation set.

**6a.** Write `choose_threshold(model, x_normal, score_fn, q)`: compute the error maps and the scores of `x_normal`, and return their `q`-quantile (with `q = 0.95`, 95% of normal images fall below the threshold, so about 5% of normal images will raise a **false alarm**).

**6b.** Write `detection_report(scores, labels, threshold)`: predict "anomaly" when `score > threshold`, and return a dictionary with the **detection rate** (fraction of anomalies flagged) and the **false alarm rate** (fraction of normal images flagged).

In [ ]:
def choose_threshold(model, x_normal, score_fn, q=0.95):
    # YOUR CODE HERE
    raise NotImplementedError()


def detection_report(scores, labels, threshold):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# ---- check step 5 ----
v1 = lambda sq, sg: score_mean(sq)          # all score functions take (sq_error, signed)
thr_v1 = choose_threshold(model, x_val, v1)
rep_v1 = detection_report(scores_v1, y_test, thr_v1)
print(f"threshold {thr_v1:.5f} -> detects {rep_v1['detection_rate']:.1%} of anomalies, "
      f"false alarms on {rep_v1['false_alarm_rate']:.1%} of normal images")
assert 0.0 < rep_v1["false_alarm_rate"] < 0.2, "with q=0.95 about 5% of normal test images should be flagged"
plot_score_histograms(scores_v1, y_test, "Score v1 with the threshold chosen on validation normals", thr_v1)
print("Step 5 OK")

## Step 6. A better score: measure the **missing ink**, locally

Two observations from Step 3:
* normal images have a small error **everywhere along the stroke**; averaging over the whole image mixes the hole with this background error;
* our anomalies are **missing** ink: the model draws a stroke where the input has none, so $\hat x - x$ is strongly **positive** inside the hole.

Write `score_missing_ink(sq_error, signed, window)`:
1. keep only the positive part of `signed` (missing ink),
2. average it over every `window x window` neighbourhood (`F.avg_pool2d` with stride 1 and padding `window // 2`), so that one isolated noisy pixel does not count, but a whole missing piece of stroke does,
3. return, for each image, the **maximum** of this map.

In [ ]:
def score_missing_ink(sq_error, signed, window=WINDOW):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# ---- check step 6 ----
scores_v2 = score_missing_ink(sq_err, signed)
assert scores_v2.shape == (len(x_test),)
auc_v2 = roc_auc_score(y_test.numpy(), scores_v2.numpy())
thr_v2 = choose_threshold(model, x_val, score_missing_ink)
rep_v2 = detection_report(scores_v2, y_test, thr_v2)
print(f"AUC  v1 (mean squared error): {auc_v1:.3f}   v2 (missing ink): {auc_v2:.3f}")
print(f"v2 threshold {thr_v2:.3f} -> detects {rep_v2['detection_rate']:.1%} of anomalies, "
      f"false alarms {rep_v2['false_alarm_rate']:.1%}  (v1 detected {rep_v1['detection_rate']:.1%})")
assert auc_v2 > auc_v1, "the missing-ink score should separate the two classes better"
plot_score_histograms(scores_v2, y_test, f"Score v2: local missing ink (AUC {auc_v2:.3f})", thr_v2)

fpr1, tpr1, _ = roc_curve(y_test.numpy(), scores_v1.numpy())
fpr2, tpr2, _ = roc_curve(y_test.numpy(), scores_v2.numpy())
plt.figure(figsize=(4.5, 4.5))
plt.plot(fpr1, tpr1, label=f"v1 mean squared error (AUC {auc_v1:.2f})")
plt.plot(fpr2, tpr2, label=f"v2 missing ink (AUC {auc_v2:.2f})")
plt.plot([0, 1], [0, 1], "k:", label="random")
plt.xlabel("false alarm rate"); plt.ylabel("detection rate"); plt.legend(fontsize=8); plt.title("ROC curves"); plt.show()
print("Step 6 OK")

## Step 7. Showing *where* the anomaly is

The image-level threshold of Step 6 answers "is this image damaged?". To answer "**where**?", we need a threshold on **pixels**, chosen again on normal images only.

**8a.** Write `choose_pixel_threshold(model, x_normal, q)`: compute the local missing-ink map (function `local_missing_ink`, given) of the normal images, and return the `q`-quantile of **all their pixel values** (with `q = 0.995`, only 0.5% of the pixels of normal images exceed it).

**8b.** Write `show_anomaly_regions(images, signed, masks, pixel_threshold, n)`. For the first `n` images, draw:
* the input in grey,
* on top of it, in **red**, the pixels where the local missing-ink map is above `pixel_threshold`,
* the **outline of the ground-truth square** in green (use `masks`, only for damaged images).

In [ ]:
def local_missing_ink(signed, window=WINDOW):
    """The map used by score_missing_ink, before taking the maximum (given)."""
    return F.avg_pool2d(signed.clamp(min=0), window, stride=1, padding=window // 2)


def choose_pixel_threshold(model, x_normal, q=0.995):
    # YOUR CODE HERE
    raise NotImplementedError()


def show_anomaly_regions(images, signed, masks, pixel_threshold, n=8, title=""):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# ---- check step 7 ----
pix_thr = choose_pixel_threshold(model, x_val)
print(f"pixel threshold: {pix_thr:.3f}")
ia = torch.where(y_test == 1)[0][:8]
inn = torch.where(y_test == 0)[0][:8]
show_anomaly_regions(x_test[ia], signed[ia], m_test[ia], pix_thr, title="damaged digits: red = detected, green = truly erased")
show_anomaly_regions(x_test[inn], signed[inn], m_test[inn], pix_thr, title="normal digits (ideally no red)")

an = y_test == 1
detected = local_missing_ink(signed) > pix_thr
found = ((detected & (m_test > 0)).flatten(1).sum(1) > 0)[an].float().mean().item()
inside = (detected & (m_test > 0))[an].sum().item() / max(detected[an].sum().item(), 1)
specks = (detected.flatten(1).sum(1) > 0)[~an].float().mean().item()
pix_auc = roc_auc_score(m_test[an].flatten().numpy(), local_missing_ink(signed)[an].flatten().numpy())
print(f"damaged digits with red inside the erased square: {found:.1%}")
print(f"red pixels of damaged digits that are truly inside the square: {inside:.1%}")
print(f"normal digits with at least one red pixel: {specks:.1%}")
print(f"pixel-level AUC (ranking of the pixels, no threshold): {pix_auc:.3f}")
assert 0 < pix_thr < 1 and pix_auc > 0.8
print("Step 7 OK")

**Questions.**
0. Some normal digits get a few red pixels. Which `q` would remove them, and what would it cost on the damaged digits? Try 0.99 and 0.999.
1. Find a damaged digit that is **not** detected. Why is it hard? (Look at how much ink was removed, and where.)
2. Our score looks only for *missing* ink. What kind of anomaly would it miss? How would you change it?
3. Would a better autoencoder (one that reconstructs everything more precisely) be a better anomaly detector?



## Going further (optional)
* **Bigger and smaller patches.** Change `PATCH_SIZE` to 5 and 9 and re-run Steps 2 to 8. Which is harder to detect?
* **Added ink.** Write `add_patch` that paints a white square instead, and adapt the score as in question 2.
* **Semantic anomalies.** Use other digits as anomalies: `load_digit(3, train=False)` mixed with normal 5s. Which digits look most like a 5 to the model?
* **Another network.** Replace the model by the `ConvAutoencoder` of lab 05 and compare the AUC of both scores after 15 and after 30 epochs.